# Financial Data Analysis Notebook
This notebook performs comprehensive financial data analysis including data retrieval, exploratory analysis, metric calculations, visualizations, and statistical testing.

## 1. Import Required Libraries
Import necessary libraries including pandas, NumPy, matplotlib, seaborn, and yfinance for financial data retrieval.

In [7]:
import pandas as pd

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Financial Data
Load financial data from sources such as CSV files or APIs using yfinance to retrieve stock prices and financial indicators.

In [9]:
# Define date range for analysis
end_date = datetime.now()
start_date = end_date - timedelta(days=365)

# Download stock data using yfinance
tickers = ['AAPL', 'MSFT', 'GOOGL']
data = yf.download(tickers, start=start_date, end=end_date, progress=False)

# Use closing prices for analysis
close_prices = data['Adj Close']
print(f"Data shape: {close_prices.shape}")
print(f"\nData period: {start_date.date()} to {end_date.date()}")
print(f"\nFirst few rows:\n{close_prices.head()}")

KeyError: 'Adj Close'

## 3. Exploratory Data Analysis
Examine the structure and summary statistics of the financial dataset, including data types, missing values, and basic descriptive statistics.

In [ ]:
# Check data types and missing values
print("Data Info:")
print(close_prices.info())
print(f"\nMissing values:\n{close_prices.isnull().sum()}")
print(f"\nBasic Statistics:\n{close_prices.describe()}")
print(f"\nCorrelation Matrix:\n{close_prices.corr()}")

## 4. Calculate Financial Metrics
Compute key financial metrics such as returns, moving averages, volatility, and Sharpe ratio to analyze portfolio performance.

In [ ]:
# Calculate daily returns
daily_returns = close_prices.pct_change()

# Calculate moving averages
ma_50 = close_prices.rolling(window=50).mean()
ma_200 = close_prices.rolling(window=200).mean()

# Calculate volatility (annualized standard deviation)
volatility = daily_returns.std() * np.sqrt(252)

# Calculate cumulative returns
cumulative_returns = (1 + daily_returns).cumprod()

# Calculate Sharpe Ratio (assuming 2% risk-free rate)
risk_free_rate = 0.02
sharpe_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (daily_returns.std() * np.sqrt(252))

print("Daily Returns (first 5 rows):")
print(daily_returns.head())
print(f"\nAnnualized Volatility:\n{volatility}")
print(f"\nSharpe Ratio:\n{sharpe_ratio}")

## 5. Visualize Financial Trends
Create visualizations including line charts for price trends, histograms for returns distribution, and correlation heatmaps.

In [ ]:
# Plot price trends with moving averages
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price trends
for ticker in tickers:
    axes[0, 0].plot(close_prices.index, close_prices[ticker], label=ticker, linewidth=2)
axes[0, 0].set_title('Stock Price Trends')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].legend()
axes[0, 0].grid()

# Cumulative returns
for ticker in tickers:
    axes[0, 1].plot(cumulative_returns.index, cumulative_returns[ticker], label=ticker, linewidth=2)
axes[0, 1].set_title('Cumulative Returns')
axes[0, 1].set_ylabel('Return Multiple')
axes[0, 1].legend()
axes[0, 1].grid()

# Returns distribution
for ticker in tickers:
    axes[1, 0].hist(daily_returns[ticker].dropna(), bins=50, alpha=0.6, label=ticker)
axes[1, 0].set_title('Daily Returns Distribution')
axes[1, 0].set_xlabel('Daily Return')
axes[1, 0].legend()
axes[1, 0].grid()

# Correlation heatmap
sns.heatmap(close_prices.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1], cbar_kws={'label': 'Correlation'})
axes[1, 1].set_title('Price Correlation Matrix')

plt.tight_layout()
plt.show()

## 6. Statistical Analysis
Perform statistical tests and analysis such as correlation analysis, risk assessment, and trend identification.

In [ ]:
# Summary statistics for returns
returns_summary = pd.DataFrame({
    'Mean Daily Return': daily_returns.mean(),
    'Std Dev (Daily)': daily_returns.std(),
    'Annual Return': daily_returns.mean() * 252,
    'Annual Volatility': daily_returns.std() * np.sqrt(252),
    'Sharpe Ratio': sharpe_ratio,
    'Min Return': daily_returns.min(),
    'Max Return': daily_returns.max(),
    'Skewness': daily_returns.skew(),
    'Kurtosis': daily_returns.kurtosis()
})

print("Returns Analysis Summary:")
print(returns_summary.round(4))

# Correlation analysis
print("\n\nPrice Correlation Matrix:")
print(close_prices.corr().round(4))

# Risk metrics
print("\n\nRisk Metrics:")
print(f"Value at Risk (VaR) at 95% confidence:")
for ticker in tickers:
    var_95 = daily_returns[ticker].quantile(0.05)
    print(f"  {ticker}: {var_95:.4f}")